# LangChain workflow: классификация жалоб

1. **Этап 0a**: из текста обращения извлекаются списки `issues` и `requested_actions`.
2. **Этап 0b**: из собранных формулировок строится черновая двухуровневая таксономия.
3. **Этап 1**: выполняется классификация по утвержденным справочникам.
4. **Этап 2**: запускаются две независимые judge-цепочки:
   - judge для `issues`
   - judge для `requested_actions`

- В LLM передается только текст обращения (`text`).
- Для длинных прогонов используется батч-обработка, чанки сохраняются в `results/`.

## Мини-справка по LangChain 

- `ChatPromptTemplate` — шаблон промпта с переменными (`{text}`, `{issues_table}` и т.д.).
- `prompt | llm` — LCEL-цепочка: сначала формируется текст prompt, затем вызывается модель.
- `with_structured_output(PydanticModel)` — просим модель вернуть структуру по схеме и валидируем ответ.
- `chain.invoke(...)` — один запрос.
- `chain.batch([...])` — много запросов пачкой.

В этом проекте логика вынесена в `complaints_workflow_pipeline.py`, а notebook отвечает за сценарий запуска.

In [ ]:
from pathlib import Path

import pandas as pd

# Импортируем готовые строительные блоки пайплайна.
# Логика уже вынесена в complaints_workflow_pipeline.py,
# поэтому в notebook мы в основном оркестрируем шаги.
from complaints_workflow_pipeline import (
    build_judge_issues_chain,
    build_judge_legacy_chain,
    build_judge_requests_chain,
    build_llm_from_config,
    build_stage0_extract_chain,
    build_stage0_taxonomy_chain,
    build_stage1_chain,
    collect_problem_phrases,
    compute_judge_metrics,
    explode_predictions,
    load_pair_config,
    read_product_context,
    run_judge,
    run_legacy_judge,
    run_stage0_extract,
    run_stage0_taxonomy,
    run_stage1_classification,
)

In [ ]:
ROOT = Path.cwd()
CONFIG_PATH = ROOT / "configs" / "credit_card_retail.json"

# Загружаем единый JSON-конфиг:
# - параметры пары продукт/категория
# - путь к входному файлу
# - пути к prompt-файлам и справочникам
# - настройки LLM
# - параметры батчинга и retry
cfg = load_pair_config(CONFIG_PATH)

print("Ключ пары:", cfg.pair_key)
print("Колонка с текстом:", cfg.text_column)
print("Входной файл:", cfg.input_path)
print("\nПуть к базовой таксономии issues:", cfg.taxonomy_issues_path)
print("Путь к базовой таксономии requested_actions:", cfg.taxonomy_requests_path)
print("\nМодель LLM:", cfg.llm.model)
print("Таймаут LLM:", cfg.llm.timeout_seconds)
print("Размер батча:", cfg.runtime.batch_size)
print("Параллельных запросов:", cfg.runtime.max_concurrency)

In [ ]:
# 1) Создаем LLM-клиент из JSON-конфига.
llm = build_llm_from_config(cfg.llm)

# 2) Берем runtime-настройки в отдельные переменные.
batch_size = cfg.runtime.batch_size
max_concurrency = cfg.runtime.max_concurrency
retries = cfg.runtime.retries
backoff_base_seconds = cfg.runtime.backoff_base_seconds

# 3) Читаем контекст продукта из product_context.txt.
# Он автоматически подставляется во все промпты через {product_context}.
product_context = read_product_context(cfg)
print("Контекст продукта:", product_context[:120] if product_context else "(не задан)")

# 4) Собираем цепочки для этапа 0.
stage0_extract_chain = build_stage0_extract_chain(
    llm,
    cfg.prompts_dir / "stage0_extract.txt",
    product_context=product_context,
)

stage0_taxonomy_chain = build_stage0_taxonomy_chain(
    llm,
    cfg.prompts_dir / "stage0_taxonomy.txt",
    product_context=product_context,
)

legacy_judge_chain = build_judge_legacy_chain(
    llm,
    cfg.prompts_dir / "judge_legacy.txt",
)

print("Цепочки этапа 0 готовы")

In [ ]:
# Путь к входному файлу берется из JSON-конфига.
INPUT_PATH = cfg.input_path

if INPUT_PATH.suffix.lower() == ".parquet":
    df = pd.read_parquet(INPUT_PATH)
else:
    df = pd.read_csv(INPUT_PATH)

# Если в таблице есть колонки product_id/product_category,
# фильтруем только нужную пару из конфига.
if "product_id" in df.columns and "product_category" in df.columns:
    df = df[
        (df["product_id"] == cfg.product_id)
        & (df["product_category"] == cfg.product_category)
    ].copy()

if cfg.text_column not in df.columns:
    raise ValueError(f"Колонка '{cfg.text_column}' не найдена во входных данных.")

# Важно: в LLM передаем только текст обращения.
# product_id/product_category оставляем только для отчетов и метрик.
working_df = df[[cfg.text_column]].copy()
working_df = working_df.rename(columns={cfg.text_column: "text"})
working_df["product_id"] = cfg.product_id
working_df["product_category"] = cfg.product_category

# Для тестового прогона — ограничить количество строк (None = все строки).
SAMPLE_SIZE = 100
if SAMPLE_SIZE is not None:
    working_df = working_df.head(SAMPLE_SIZE)

print("Строк для обработки:", len(working_df))
working_df.head(2)

In [ ]:
# Этап 0a: извлекаем "сырые" проблемы и запросы из текста.
# Результат сохраняется чанками в results/.../stage0_extract/*.parquet
stage0_out_dir = ROOT / "results" / cfg.pair_key / "stage0_extract"
stage0_df = run_stage0_extract(
    source_df=working_df,
    chain=stage0_extract_chain,
    output_dir=stage0_out_dir,
    text_column="text",
    batch_size=batch_size,
    max_concurrency=max_concurrency,
    retries=retries,
    base_sleep_seconds=backoff_base_seconds,
)

print("Этап 0a, строк обработано:", len(stage0_df))
stage0_df[["text", "issues_raw", "requested_actions_raw"]].head(2)

In [ ]:
# Этап 0b: строим две отдельные таксономии (issues и requested_actions).
# Для обеих используем ОДИН шаблон prompt: stage0_taxonomy.txt
issues_phrases = collect_problem_phrases(stage0_df, columns=["issues_raw"])
requests_phrases = collect_problem_phrases(stage0_df, columns=["requested_actions_raw"])

generated_issues_taxonomy_path = ROOT / "results" / cfg.pair_key / "stage0_taxonomy_issues.json"
generated_requests_taxonomy_path = ROOT / "results" / cfg.pair_key / "stage0_taxonomy_requested_actions.json"

generated_issues_taxonomy_md = run_stage0_taxonomy(
    chain=stage0_taxonomy_chain,
    problems=issues_phrases,
    output_path=generated_issues_taxonomy_path,
    retries=retries,
    base_sleep_seconds=backoff_base_seconds,
)

generated_requests_taxonomy_md = run_stage0_taxonomy(
    chain=stage0_taxonomy_chain,
    problems=requests_phrases,
    output_path=generated_requests_taxonomy_path,
    retries=retries,
    base_sleep_seconds=backoff_base_seconds,
)

print("Таксономия issues сохранена в:", generated_issues_taxonomy_path)
print("Таксономия requested_actions сохранена в:", generated_requests_taxonomy_path)
print("Фраз issues:", len(issues_phrases), "| Фраз requested_actions:", len(requests_phrases))
print("\nПредпросмотр таксономии issues:\n")
print(generated_issues_taxonomy_md[:800])
print("\nПредпросмотр таксономии requested_actions:\n")
print(generated_requests_taxonomy_md[:800])

In [ ]:
# ВАЖНО: на этом шаге можно вручную поправить таксономии прямо в папке results.
# Файлы для редактирования:
# - generated_issues_taxonomy_path
# - generated_requests_taxonomy_path
# После ручной правки просто запускай эту ячейку: она перечитает файлы и
# пересоберет цепочки этапа 1 и judge на основе финального текста в этих файлах.

print("Проверь и при необходимости отредактируй файлы:")
print("-", generated_issues_taxonomy_path)
print("-", generated_requests_taxonomy_path)

In [ ]:
issues_table = generated_issues_taxonomy_path.read_text(encoding="utf-8")
requests_table = generated_requests_taxonomy_path.read_text(encoding="utf-8")

stage1_chain = build_stage1_chain(
    llm,
    cfg.prompts_dir / "stage1_classification.txt",
    issues_table=issues_table,
    requests_table=requests_table,
    product_context=product_context,
)
judge_issues_chain = build_judge_issues_chain(
    llm,
    cfg.prompts_dir / "judge_issues.txt",
    issues_table=issues_table,
    product_context=product_context,
)
judge_requests_chain = build_judge_requests_chain(
    llm,
    cfg.prompts_dir / "judge_requests.txt",
    requests_table=requests_table,
    product_context=product_context,
)

print("Цепочки перечитаны из файлов в results/. Можно запускать этап 1.")

In [ ]:
# Этап 1: классифицируем каждое обращение строго по справочнику.
stage1_out_dir = ROOT / "results" / cfg.pair_key / "stage1_classification"
classified_df = run_stage1_classification(
    source_df=working_df,
    chain=stage1_chain,
    output_dir=stage1_out_dir,
    text_column="text",
    batch_size=batch_size,
    max_concurrency=max_concurrency,
    retries=retries,
    base_sleep_seconds=backoff_base_seconds,
)

print("Этап 1, строк обработано:", len(classified_df))
classified_df[["text", "issues_pred", "requested_actions_pred"]].head(2)

In [ ]:
# Перед judge разворачиваем списки меток в плоские таблицы:
# одна строка = один присвоенный label.
issues_eval_df = explode_predictions(classified_df, labels_column="issues_pred", text_column="text")
requests_eval_df = explode_predictions(
    classified_df,
    labels_column="requested_actions_pred",
    text_column="text",
)

issues_judge_out_dir = ROOT / "results" / cfg.pair_key / "judge_issues"
requests_judge_out_dir = ROOT / "results" / cfg.pair_key / "judge_requests"

# Judge 1: проверяем корректность проблем (issues).
issues_judged_df = run_judge(
    labels_df=issues_eval_df,
    chain=judge_issues_chain,
    output_dir=issues_judge_out_dir,
    batch_size=batch_size,
    max_concurrency=max_concurrency,
    retries=retries,
    base_sleep_seconds=backoff_base_seconds,
    file_prefix="judge_issues",
)

# Judge 2: отдельно проверяем корректность запросов (requested_actions).
requests_judged_df = run_judge(
    labels_df=requests_eval_df,
    chain=judge_requests_chain,
    output_dir=requests_judge_out_dir,
    batch_size=batch_size,
    max_concurrency=max_concurrency,
    retries=retries,
    base_sleep_seconds=backoff_base_seconds,
    file_prefix="judge_requests",
)

print("Проверено judge (issues), строк:", len(issues_judged_df))
print("Проверено judge (requested_actions), строк:", len(requests_judged_df))

In [ ]:
# Считаем метрики качества:
# - общий процент корректных category/sub_category
# - разрез по product_id/product_category
issues_metrics = compute_judge_metrics(
    issues_judged_df,
    group_columns=["product_id", "product_category"],
)
requests_metrics = compute_judge_metrics(
    requests_judged_df,
    group_columns=["product_id", "product_category"],
)

display(issues_metrics)
display(requests_metrics)

# Опционально: legacy judge, если в данных есть колонка со старой плоской меткой.
# if "legacy_label" in working_df.columns:
#     legacy_source = working_df[["text", "legacy_label"]].rename(columns={"legacy_label": "label"})
#     legacy_out = ROOT / "results" / cfg.pair_key / "judge_legacy"
#     legacy_df = run_legacy_judge(
#         labels_df=legacy_source,
#         chain=legacy_judge_chain,
#         output_dir=legacy_out,
#         label_column="label",
#         text_column="text",
#         batch_size=batch_size,
#         max_concurrency=max_concurrency,
#         retries=retries,
#         base_sleep_seconds=backoff_base_seconds,
#     )
#     display(legacy_df.head())

## Чеклист перед запуском в прод

- Проверены пути для пары `product_id/product_category` (`prompts`, `results`).
- В prompt передается только `text`.
- Этап 0b генерирует две отдельные таксономии: issues и requested_actions.
- Перед этапом 1 при необходимости вручную отредактированы файлы в `results/<pair>/`.
- После ручной правки запущена ячейка, которая перечитывает эти файлы и пересобирает цепочки.
- Таймаут LLM не выше 60 секунд.
- Длинные прогоны разбиты на батчи и сохраняются чанками в parquet.
- Judge по `issues` и `requested_actions` запускается независимо.
- Метрики считаются глобально и по разрезам продукта/категории.